# Chuckles on RunPod

Use the 27B Qwen GGUF to write three or four movie-title parodies for an audience that likes edgy humor. The model pitches jokes from the supplied word list and nominates two finalists before it calls the verifier. It then checks every word with the two tools from the [original Colab cell](https://colab.research.google.com/drive/1j6x-MpKF-Qm2hV1YDZfz7AOCc50PogX1#scrollTo=GvGq8GcoGbCD).

This notebook and the command-line launcher use the same CodeAgent pipeline. Keep secrets in the Colab Secrets panel or environment variables. A CPU notebook runtime is sufficient.

In [ ]:
import inspect
import json
import os
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

here = Path.cwd()
root = next((d for d in (here, *here.parents) if (d / "pipeline/run_gguf_test.py").is_file()), here / "chuck26")
if not root.is_dir():
    subprocess.run(["git", "clone", "--depth=1", "https://github.com/patruff/chuck26.git", str(root)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(root) + "[trace-export]",
                "smolagents==1.24.0", "huggingface-hub==0.36.0", "setuptools<81"], check=True)
launcher = str(root / "pipeline/run_gguf_test.py")

## Check the word bank on CPU

The word bank lives in the [private 2025 repository](https://github.com/patruff/chuck2025/blob/783c52912c0be25ccc27fbca768548c46c977a29/word_structures.py). On hosted Colab, upload that file and set `WORD_BANK_PATH` to its path, or set the Colab secret `CHUCK2025_READ_TOKEN` to a token with read access to the source repository. Local users can also use an authenticated GitHub CLI or a sibling `chuck2025` checkout.

The check verifies the file checksum, the complete Running Man suggestions, and both phonetic scores. It checks that *hunning* is unscorable even though *huning* scores 0.83.

In [ ]:
WORD_BANK_PATH = ""  # @param {type:"string"}
bank_args = ["--march-word-bank", WORD_BANK_PATH] if WORD_BANK_PATH else []
env = os.environ.copy()

def load_colab_secret(name):
    if name in env or "google.colab" not in sys.modules:
        return
    from google.colab import userdata
    try:
        env[name] = userdata.get(name)
    except userdata.SecretNotFoundError:
        pass

if not bank_args:
    load_colab_secret("CHUCK2025_READ_TOKEN")
subprocess.run([sys.executable, launcher, "--preflight-only", *bank_args], cwd=root, env=env, check=True)

## Creative and verification prompt

The runner reads the prompt from this cell. The agent reads the returned words first and nominates its funniest supplied-word pitch when one makes sense. A winner needs a funny premise and a verifiable pronunciation for its actual spelling.

In [ ]:
def generate_parody(title, suggestions):
    prompt = f"""Create funny parodies of the title '{title}'

CREATIVE PHASE:
First, generate 3-4 different funny variations without worrying about phonetics.
For each attempt:
1. Try different approaches (changing 1 word, 2 words, etc.)
2. Focus purely on humor and clever wordplay
3. The audience likes edgy, dark, sexual, violent, adult themes.
4. Explain why each variation is funny or falls flat. Make the joke audible in the title; describe the comic scene it creates. Reject a respelling with no comprehensible joke.
5. Rate each variation on a humor scale of 1-10

VERIFICATION PHASE:
After generating variations:
1. Take your top 2 funniest variations
2. For EACH WORD in each variation:
   - Use word_phone_tool to check similarity with the original word
   - Display the March suggestion similarity and the separate word-phone score. Verify the exact spelling printed in the title.
   - Each changed word MUST have an observed parody-suggestions similarity >= 0.60
3. Example verification:
   Original: "word1 word2 word3"
   Parody: "word1 wurd2 word3"
   Checking each pair:
   - word1 vs word1: [call word_phone_tool]
   - word2 vs wurd2: [call word_phone_tool]
   - word3 vs word3: 1.0 (identical)
4. If any word pair scores below 0.6, try alternative words
5. Choose final version that's both funny AND meets phonetic requirements

Start with these actual returned suggestions. Pitch the funniest changes they set up in this movie. Compare them before you verify your finalists:
{json.dumps(suggestions, indent=2)}

<think>
Present your variations with this exact format:
### Attempt 1:
**"[Your first variation]"**
Humor rating: X/10
Why it's funny: [explanation]

### Attempt 2:
**"[Your second variation]"**
Humor rating: X/10
Why it's funny: [explanation]

### Attempt 3:
**"[Your third variation]"**
Humor rating: X/10
Why it's funny: [explanation]

Then verify your top choices:
1. Test EACH WORD with word_phone_tool
2. Show all similarity scores
3. Check that each changed word has observed March similarity >= 0.60
4. Make final selection
</think>

### Final Chosen Parody:
**"[Your chosen parody]"**

### Word-by-Word Phonetic Verification:
[Show each word pair and its similarity score]
- Original word 1 vs Parody word 1: [score]
- Original word 2 vs Parody word 2: [score]
[etc.]

### Final Reasoning:
1. Why this version was chosen
2. Observed March similarity for each change and any shortfall below 0.60
3. How it balances humor and sound"""
    return prompt

## Run one title or a batch

Set `RUNPOD_API_KEY` in Colab Secrets or the environment. Running the next cell rents a RunPod GPU. The default uses the first title in the checked-in sample. Set `LIMIT` to 10 for the full batch. You can edit the prompt cell before running.

In [ ]:
LIMIT = 1  # @param {type:"integer"}
load_colab_secret("RUNPOD_API_KEY")
run_dir = root / "out" / ("colab-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ"))
run_dir.parent.mkdir(parents=True, exist_ok=True)
# Pass the live prompt cell, including edits made in this notebook session.
import tempfile
with tempfile.TemporaryDirectory() as temp:
    notebook = Path(temp) / "active-prompt.ipynb"
    notebook.write_text(json.dumps({"cells": [{"cell_type": "code", "metadata": {"id": "GvGq8GcoGbCD"},
                                               "source": inspect.getsource(generate_parody)}]}), encoding="utf-8")
    subprocess.run([sys.executable, launcher, "--humor-first", "--example-notebook", str(notebook),
                    *bank_args, "--limit", str(LIMIT), "--output-dir", str(run_dir)],
                   cwd=root, env=env, check=True)

## Read the results

The March suggestion score sets the inclusive 0.60 floor. Word-phone supplies an additional rhyme, phone-count and stress check. Review the model’s humor judgments separately. The JSONL keeps the drafts, thinking, executed Python and tool observations.

In [ ]:
rows = [json.loads(line) for line in (run_dir / "results.jsonl").read_text().splitlines()]
for row in rows:
    print(row["input_title"], "→", row.get("qualifying_parody") or "No qualifying winner")
    if row.get("fallback"):
        print("  Preferred fallback:", row["fallback"])
    for title, pairs in row.get("finalist_checks", {}).items():
        print(" ", title, [(p["word"], p["compare_to"], p["suggestion_score"], p["word_phone_score"])
                           for p in pairs if p["word"] != p["compare_to"]])
candidates = run_dir / "candidates.jsonl"
if not candidates.exists():
    with tempfile.TemporaryDirectory(dir=run_dir) as temp:
        export = Path(temp) / "candidates.jsonl"
        subprocess.run([sys.executable, str(root / "pipeline/export_humor_candidates.py"),
                        str(run_dir / "results.jsonl"), str(export),
                        "--run-manifest", str(run_dir / "run.json")], cwd=root, env=env, check=True)
        export.replace(candidates)
print("Review-pending turns:", len(candidates.read_text().splitlines()), "in", candidates)
print("Complete Qwen thinking and tool transcripts:", run_dir / "TRACE-INDEX.md")
from IPython.display import Markdown, display
first_trace = next(iter(sorted((run_dir / "transcripts").glob("*.md"))), None)
if first_trace:
    display(Markdown(first_trace.read_text()))